# 06 · Distillation: DistilBERT learns from DeBERTa-v3

**Experiment 2 of `qa-experiments`.** Train DistilBERT again with the same settings as our first one
(64.1 EM / 67.4 F1), but this time it learns from two things: the SQuAD labels as before, and the scores of our
DeBERTa-v3 (84.9 / 87.7), the "teacher". The teacher gives a score to every possible start and end, not just
the right one, so the student also learns which answers were *nearly* right and how sure to be about "no
answer". The goal: a model as small and fast as DistilBERT, but more accurate. How it works: `src/distill.py`.

Run it in Google Colab with **Runtime → Change runtime type → T4 GPU**.

1. **Trial run** (sections 0–1, about 5 minutes): the teacher scores 1% of the training windows, then 50
   training steps. Checks that it all works and estimates the time of the full run.
2. **Full run** (sections 2–4): the teacher scores all 131,000 windows once (saved to Drive), then 2 epochs of
   training, the upload to the Hub as `Ingsyling/distilbert-squad2-distilled`, and the dev-set score.

**Do the trial run first and send its output before starting the full run.**

All settings are in `configs/distill.yaml`.

## 0. Setup
Clones this repo and installs the requirements. Safe to run more than once.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Davidfaleye360/qa-experiments.git"
REPO_DIR = "/content/qa-experiments"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import transformers, datasets, torch

print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError("transformers 5.x is still loaded. Use Runtime -> Restart session, then run the setup cell again.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then run the setup cell again.")
    print("The smoke test still works without one, just more slowly.")

## 1. Trial run

`configs/smoke.yaml` on top of `configs/distill.yaml`: 1% of the training data, 50 steps. What to look for:

- **`Teacher scored ... windows`**: how fast DeBERTa scores windows. The full run needs about 131,000.
- **`Check: the teacher's top start matches the SQuAD label in ...%`**: about 80% is expected (on this Mac:
  82% start, 86% end, 77% both). Most disagreements are the teacher choosing a slightly longer or shorter span
  ("the Carpathian Basin" for "Carpathian Basin"). A number near 0 would mean the scores are lined up wrong.
- **The loss starts near 10, not 5.9 as before.** It now adds the difference from the teacher, which is large
  at the start. It should fall, and the run should end with **RESULT: PASS**.

In [ ]:
!{sys.executable} -m src.train --config configs/distill.yaml --overrides configs/smoke.yaml --fresh

What the trial says about the full run: a time estimate for the teacher and for training.

In [ ]:
import glob, json, math
from src.config import load_config

full = load_config("configs/distill.yaml")
trial = json.load(open("outputs/smoke/train_summary.json"))
timing = json.load(open(sorted(glob.glob("outputs/smoke/teacher_scores_*/teacher_timing.json"))[-1]))

windows_full = trial["train_windows"] / 0.01                    # the trial used 1% of the data
teacher_minutes = windows_full / (timing["windows"] / timing["seconds"]) / 60
steps_full = math.ceil(windows_full / full.training.batch_size) * full.training.num_epochs
train_minutes = steps_full * trial["runtime_seconds"] / trial["steps"] / 60

print(f"Teacher: {timing['windows']:,} windows in {timing['seconds']:.0f} s -> all of them in about {teacher_minutes:.0f} minutes")
print(f"Training: {trial['steps']} steps in {trial['runtime_seconds']:.0f} s, peak GPU memory {trial.get('peak_gpu_memory_gb')} GB "
      f"-> {steps_full:,} steps in about {train_minutes:.0f} minutes")
print(f"Losses: {[round(loss, 2) for _, loss in trial['losses']]}")
print(f"Full run: roughly {(teacher_minutes + train_minutes) / 60:.1f} hours (the trial includes start-up time, so this is on the high side)")
print("RESULT:", "PASS" if trial["passed"] else "FAIL")

---
# Full run

## 2. Before you start

1. **The trial passed.**
2. **Hugging Face token:** the **key icon** in Colab's left sidebar must have the secret `HF_TOKEN` (Write
   permission) with "Notebook access" on, as before. Never paste the token into a cell. The upload creates
   a new public model page, `Ingsyling/distilbert-squad2-distilled`; the group's `Ingsyling/distilbert-squad2`
   is not touched.
3. **Google Drive:** about **3 GB free**: the teacher's scores (under 1 GB) and two DistilBERT checkpoints
   (about 800 MB each). Replaced checkpoints go to the Drive trash, which still counts.
4. **Keep the tab open.**

**If Colab disconnects:** reconnect, run section 0 again, then sections 2 and 3. The teacher's scores are saved
on Drive once they are finished, and training continues from the newest checkpoint. A disconnect while the
teacher is still scoring means scoring starts again.

In [ ]:
import shutil

RUN_DIR = "/content/drive/MyDrive/qa-experiments/distilbert-squad2-distilled"   # teacher scores, checkpoints, final model

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # src/train.py reads the token from here
else:
    RUN_DIR = "outputs/distilbert-squad2-distilled"

os.makedirs(RUN_DIR, exist_ok=True)
print(f"Saving to: {RUN_DIR}")
print(f"Free space there: {shutil.disk_usage(RUN_DIR).free / 1e9:.1f} GB (the run needs about 3)")
print("Hugging Face token found:", bool(os.environ.get("HF_TOKEN")))
print("Teacher scores already there:", sorted(p for p in os.listdir(RUN_DIR) if p.startswith("teacher_scores")) or "none")
print("Checkpoints already there:", sorted(p for p in os.listdir(RUN_DIR) if p.startswith("checkpoint-")) or "none (a new run)")

## 3. Train

All of `configs/distill.yaml`. It first checks that it can upload to the Hub, so a bad token fails in seconds.
Then the teacher scores every training window (a progress bar, "Teacher scores"), then training starts; the
loss is printed every 100 steps. It ends with **RESULT: PASS** and
`Uploaded to https://huggingface.co/Ingsyling/distilbert-squad2-distilled`.

No `--fresh` on purpose: running this cell again continues an interrupted run.

In [ ]:
!{sys.executable} -m src.train --config configs/distill.yaml --output-dir "{RUN_DIR}" --push-to-hub

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

run = json.load(open(f"{RUN_DIR}/train_summary.json"))
curve = pd.DataFrame(run["losses"], columns=["step", "loss"]).set_index("step")
ax = curve["loss"].plot(color="#2a78d6", alpha=0.35, figsize=(8, 3.5), label="every 100 steps")
curve["loss"].rolling(10, min_periods=1).mean().plot(ax=ax, color="#2a78d6", label="average of the last 10")
ax.set_title(f"Distilled DistilBERT training loss ({run['steps']:,} steps)")
ax.set_xlabel("step"); ax.set_ylabel("loss (labels + teacher)"); ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False); plt.tight_layout(); plt.savefig(f"{RUN_DIR}/loss_curve.png", dpi=150); plt.show()
print(f"Steps: {run['steps']:,} | last loss: {curve['loss'].iloc[-1]:.3f} | RESULT:", "PASS" if run["passed"] else "FAIL")

## 4. Score it on the dev set

All 11,873 dev questions with the same script as before (a few minutes on a T4). To compare with: our first
DistilBERT **64.14 EM / 67.38 F1**, RoBERTa 79.90 / 82.91, the teacher 84.86 / 87.69.

In [ ]:
!{sys.executable} -m src.evaluate --model "{RUN_DIR}" --name distilbert-squad2-distilled --output-dir "{RUN_DIR}/results"

## Next

Send the output of the trial (section 1), and after the full run the end of section 3 and the score table from
section 4. The scores then go into the repo, and the speed test, threshold sweep and error analysis can be
rerun with the new model.